# FedProx μ Retry — Ceftriaxone × *E. coli*

**Re-run only FedProx with a new μ, then update the last run's results in-place.**

Set `TARGET_RUN` and `RETRY_MU` below, then run all cells.

| What's reloaded | What's re-run |
|---|---|
| Data + preprocessing | FedProx with new μ only |
| Centralized MLP/RF baselines | Convergence plot + FedProx per-site plot |
| FedAvg, FedLR, FedRF per-round CSVs | Heatmaps + final_results.csv |
| All best params | Models saved to run's models/ directory |

The target run's existing files are overwritten with updated results.

Re-run utility for the **FedProx** arm of a completed Ceftriaxone × *E. coli*
federated run: it writes the requested proximal weight ``RETRY_MU`` into the
strategy, retrains, and overwrites only the FedProx results while keeping the
other strategies' reloaded histories.

Reference implementation for the documentation page
[docs/06b](https://asd-standard.github.io/Dryad_DRIAMS/06b-Ceftriaxone-E-coli/index.html).

## Notes

- **Inputs** — a completed run directory (``TARGET_RUN``) containing
  ``results/best_params_used.txt``, the other strategies' per-round CSVs and
  ``final_results.csv``; spectra are re-read from the processed Ceftriaxone
  CSVs, *E. coli* only.
- **Outputs** — overwritten ``results/fedprox_mu{RETRY_MU}_per_round.csv`` and
  ``results/final_results.csv``, refreshed convergence/heatmap PDFs, and
  per-round FedProx checkpoints under ``models/fedprox_mlp_mu{RETRY_MU}/``.
- **Why it exists** — μ only affects the FedProx arm, so the rest of the run
  is reloaded instead of recomputed; ``NUM_ROUNDS=30`` matches the original
  run to keep the curves comparable.

In [ ]:
# ── CONFIG ──
# TARGET_RUN selects the completed run to update; RETRY_MU is the new
# FedProx proximal weight; NUM_ROUNDS should match the original run (30) so
# the curves stay comparable.
TARGET_RUN = "03-Run"   # <-- which run to update
RETRY_MU   = 0.1        # <-- new FedProx mu value
NUM_ROUNDS = 30         # use 30 to match the original

In [ ]:
# Colab dependency install; locally the same stack comes from
# requirements.txt / environment.yml (Flower + MaldiDeepKit +
# MaldiAMRKit).
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
# Mounts Google Drive on Colab; on a local machine DRYAD is
# resolved directly (see next cell).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# Imports (Flower simulation, torch MLP + MaldiDeepKit's
# SpectralAttentionMLP, sklearn baselines, plotting), the shared
# seed and the CPU/GPU device used by the retried FedProx arm.
import warnings, os, io, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch; import torch.nn as nn; import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
import flwr as fl; import joblib
warnings.filterwarnings("ignore")
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
FL_DEVICE = "cpu"; DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Flower: {fl.__version__}  |  Target: {TARGET_RUN}  |  mu: {RETRY_MU}")

In [ ]:
# Resolves the Dryad root (Colab vs. local) and the run directory
# (TARGET_RUN) whose FedProx arm is re-run.
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"; DRUG_CSV = "Ceftriaxone"; SPECIES = "Escherichia coli"
PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06-Ceftazidime-E-coli"
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"; MODEL_DIR = RUN_DIR / "models"
OUT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Run dir: {RUN_DIR}")
print(f"Drug: {DRUG_NAME}")

In [ ]:
# Reloads the frozen MLP/RF hyperparameters (learning rate,
# dropout, thresholds) from the run's best_params_used.txt;
# FedProx reuses the original MLP configuration.
params_path = OUT_DIR / "best_params_used.txt"
if not params_path.exists(): raise FileNotFoundError(f"{params_path} not found")
params = {}
with open(params_path) as f:
    for line in f:
        if "=" in line: k, v = line.strip().split("=", 1); params[k] = v
BEST_MLP_LR = float(params["BEST_MLP_LR"]); BEST_MLP_DH = float(params["BEST_MLP_DH"])
BEST_MLP_THRESH = float(params["BEST_MLP_THRESH"]); BEST_RF_THRESH = float(params["BEST_RF_THRESH"])
RF_PARAMS = eval(params["RF_PARAMS"])
print(f"MLP: lr={BEST_MLP_LR:.2e}  dropout={BEST_MLP_DH:.1f}  thresh={BEST_MLP_THRESH:.3f}")
print(f"RF:  {RF_PARAMS}  thresh={BEST_RF_THRESH:.3f}")

In [ ]:
# Threshold grid for BalAcc reporting, the MLP batch size and the
# per-site n_mixes schedule: each client averages several
# independent local passes per round, fewer as the site grows.
THRESHOLDS = np.linspace(0.05, 0.95, 91)
BATCH_SIZE = 16
def n_mixes(n_train):
    """Number of local averaged passes per FedProx round for a site.

    Parameters
    ----------
    n_train : int
        Number of local training spectra.

    Returns
    -------
    int
        3 for small sites (<1500 samples), 2 below 5000, and 1 otherwise;
        the passes are averaged by ``train_local_mixed`` before returning.
    """
    if n_train < 1500: return 3
    if n_train < 5000: return 2
    return 1

In [ ]:
# Reads each site's processed Ceftriaxone CSV, filters to E. coli
# and stores the 6000-bin matrix plus the binary resistance label;
# prints the per-site susceptible/resistant split.
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32"); y = df_eco["label"].to_numpy(dtype="int64")
    raw_data[site] = (X, y)
    nr, ns = (y==1).sum(), (y==0).sum()
    print(f"  Site {site}: {len(y)} samples ({ns} S, {nr} R)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")

In [ ]:
# Per-site 90/10 label-stratified split with the original per-site
# seeds (A=42, B=123, C=456, D=789), reproducing the split used by
# the run so the retried FedProx curve is comparable to the
# stored one.
client_train, client_test = {}, {}
site_seeds = {"A":42,"B":123,"C":456,"D":789}
for site in SITE_ORDER:
    X, y = raw_data[site]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.10, stratify=y, random_state=site_seeds[site])
    client_train[site] = (X_tr, y_tr); client_test[site] = (X_te, y_te)
    print(f"  Site {site}: train={len(X_tr)} test={len(X_te)}")

In [ ]:
# Per-site log1p+standardize fitted on each client's train split
# only and applied to its test split (leak-safe, per-site
# scaling).
client_train_pp, client_test_pp = {}, {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]; X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done.")

In [ ]:
# Model utilities: a torch Dataset wrapper, the MLP factory
# (6000-512-256-128 with the loaded dropout) and the
# state_dict <-> numpy conversion helpers used by the client.
class DatasetFromNumpy(Dataset):
    """Minimal ``torch.utils.data.Dataset`` over preprocessed numpy arrays."""
    def __init__(self, X, y):
        """Convert ``X`` and ``y`` to float32/long tensors."""
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        """Number of samples in the dataset."""
        return len(self.X)
    def __getitem__(self, i):
        """Return the ``i``-th ``(spectrum, label)`` pair."""
        return self.X[i], self.y[i]

def build_mlp():
    """Build the shared federated MLP: 6000 -> 512 -> 256 -> 128 -> 2.

    Uses MaldiDeepKit's ``SpectralAttentionMLP`` with the loaded dropout and
    attention disabled, matching the architecture of the original run.
    """
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256,128),
        dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)

def model_to_numpy(model):
    """Flatten a model's ``state_dict`` into a list of numpy arrays (Flower format)."""
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    """Load a list of numpy arrays (Flower order) back into ``model`` in place."""
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params): sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

def mlp_predict_proba(model, X_np, dev="cpu"):
    """Resistant-class probability per row of ``X_np`` (softmax[:, 1]).

    Runs ``model`` in eval mode on ``dev`` with gradients disabled.
    """
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad(): return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# Reloads the run's stored results: FedAvg/FedLR/FedRF per-round
# histories, every existing FedProx mu history, and the
# centralized/cross-site baselines from final_results.csv, so only
# the retried mu row is recomputed.
fedavg_hist = []; fedlr_hist = []; fedrf_hist = []; fedprox_histories = {}
centralized_mlp_results = {}; centralized_rf_results = {}
cross_site_results = {}; cross_site_rf_results = {}

if (OUT_DIR / "fedavg_per_round.csv").exists():
    fedavg_hist = pd.read_csv(OUT_DIR / "fedavg_per_round.csv").to_dict("records")
    print(f"Loaded fedavg: {len(fedavg_hist)} rounds")
if (OUT_DIR / "fedlr_per_round.csv").exists():
    fedlr_hist = pd.read_csv(OUT_DIR / "fedlr_per_round.csv").to_dict("records")
    print(f"Loaded fedlr: {len(fedlr_hist)} rounds")
if (OUT_DIR / "fedrf_per_round.csv").exists():
    fedrf_hist = pd.read_csv(OUT_DIR / "fedrf_per_round.csv").to_dict("records")
    print(f"Loaded fedrf: {len(fedrf_hist)} rounds")
for f in sorted(OUT_DIR.glob("fedprox_mu*_per_round.csv")):
    mu_str = f.stem.replace("fedprox_mu","").replace("_per_round","")
    mu = float(mu_str)
    fedprox_histories[mu] = pd.read_csv(f).to_dict("records")
    print(f"Loaded fedprox mu={mu}: {len(fedprox_histories[mu])} rounds")

if (OUT_DIR / "final_results.csv").exists():
    df_old = pd.read_csv(OUT_DIR / "final_results.csv")
    for meth, target in [("Centralized MLP", centralized_mlp_results), ("Centralized RF", centralized_rf_results),
                          ("Cross-Site MLP", cross_site_results), ("Cross-Site RF", cross_site_rf_results)]:
        row = df_old[df_old["Method"] == meth]
        if len(row) > 0:
            r = row.iloc[0]
            for site in SITE_ORDER:
                for suffix in ["BalAcc","AUC"]:
                    col = f"{site}_{suffix}"
                    if col in r and not pd.isna(r[col]): target[col] = float(r[col])
            if "All_BalAcc" in r and not pd.isna(r["All_BalAcc"]): target["All_BalAcc"] = float(r["All_BalAcc"])
            if "All_AUC" in r and not pd.isna(r["All_AUC"]): target["All_AUC"] = float(r["All_AUC"])
    print(f"Loaded centralized + cross-site results")

In [ ]:
# Local training for one FedProx client: n_mixes independent
# shuffled passes at the loaded MLP learning rate; when
# proximal_mu > 0 the loss adds a (mu/2)*||w - w_global||^2 term.
# The passes' state_dicts are averaged before returning, and
# FedMLPClient bundles this behind Flower's NumPyClient.
def train_local_mixed(model, X_np, y_np, n_mixes, proximal_mu, global_params):
    """Train one client locally with the FedProx objective.

    Parameters
    ----------
    model : torch.nn.Module
        Local model, updated in place.
    X_np, y_np : ndarray
        Preprocessed local training data.
    n_mixes : int
        Number of independent shuffled passes to average (see ``n_mixes``).
    proximal_mu : float
        FedProx mu; 0 disables the proximal term.
    global_params : list of torch.Tensor or None
        Snapshot of the round's global weights used by the proximal term.

    Returns
    -------
    float
        Mean training loss across the ``n_mixes`` passes.
    """
    collect_sds = []; total_loss = 0.0
    for mix_i in range(n_mixes):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train(); batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) ** 2 for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim()==0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes

class FedMLPClient(fl.client.NumPyClient):
    """Flower NumPyClient for one site's MLP client under FedProx.

    Parameters
    ----------
    cid : str
        Client id; ``SITE_ORDER[int(cid)]`` resolves it to a DRIAMS site.
    X_train : ndarray
        Preprocessed local training spectra (6000 bins).
    y_train : ndarray
        Binary resistance labels for ``X_train``.
    """
    def __init__(self, cid, X_train, y_train):
        """Store the site data, pick the mix count and build the local MLP."""
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        self.n_mixes = n_mixes(len(X_train))
        self.model = build_mlp().to(FL_DEVICE)
    def get_parameters(self, config):
        """Return the local model's parameters as numpy arrays."""
        return model_to_numpy(self.model)
    def set_parameters(self, params):
        """Load the server's numpy parameters into the local model."""
        numpy_to_model(self.model, params)
    def fit(self, parameters, config):
        """Set the global weights and run one proximal local training pass.

        Returns
        -------
        tuple
            ``(parameters, num_examples, metrics)``; ``metrics`` carries the mean
            ``train_loss`` and the ``n_mixes`` used, which the strategy aggregates.
        """
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0: global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train, self.n_mixes, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train), {"train_loss": loss, "n_mixes": self.n_mixes})

# Flower client-fn: maps a numeric client id to its DRIAMS site and
# wraps that site's preprocessed train split in a FedMLPClient.
def mlp_client_fn(cid):
    """Build the FedProx client for site ``SITE_ORDER[int(cid)]``."""
    site = SITE_ORDER[int(cid)]
    return FedMLPClient(cid, *client_train_pp[site]).to_client()

In [ ]:
# Server side: CheckpointFedProx saves each round's client and
# global model, then aggregates with the FedProx rule;
# get_mlp_eval_fn builds the central evaluator that scores the
# global model per site and pooled and fills the history used for
# the per-round CSV.
class CheckpointFedProx(fl.server.strategy.FedProx):
    """FedProx strategy that checkpoints per-client and global models each round.

    Parameters
    ----------
    strategy_name : str
        Sub-directory name under ``model_dir`` (e.g. ``fedprox_mlp_mu0.1``).
    model_dir : str or pathlib.Path
        Run's ``models/`` directory.
    **kwargs
        Forwarded to ``fl.server.strategy.FedProx`` (including ``proximal_mu``).
    """
    def __init__(self, strategy_name, model_dir, **kwargs):
        """Create the checkpoint directory and pass the strategy options to FedProx."""
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        """Save the round's client/global models, then aggregate with FedProx.

        Returns
        -------
        tuple
            ``(aggregated_parameters, aggregation_metrics)`` from the parent class;
            ``aggregation_metrics`` includes the clients' mean ``train_loss``.
        """
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    """Build the centralized ``evaluate_fn`` for the FedProx arm.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` per-site preprocessed test splits.
    threshold : float
        Decision threshold loaded from the run's ``best_params_used.txt``.
    hist_list : list
        Accumulator receiving one metrics record per round.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config)`` returning
        ``(loss, metrics)``; metrics are also written next to the round's
        checkpoints when that directory exists.
    """
    def evaluate(server_round, parameters, config):
        """Score the round's global MLP on all four sites.

        Returns
        -------
        tuple
            ``(1 - pooled BalAcc, record)``; Flower minimises the first element.
        """
        m = build_mlp(); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = mlp_predict_proba(m, X_tt, dev=FL_DEVICE)
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        rd = Path(MODEL_DIR) / f"fedprox_mlp_mu{RETRY_MU}" / f"round_{server_round:03d}"
        if rd.exists():
            with open(rd / "metrics.json", "w") as f: json.dump(record, f)
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

In [ ]:
# Runs the retried FedProx arm: 4 clients (one per site), RETRY_MU
# as the proximal weight and NUM_ROUNDS rounds of in-process
# Flower simulation; the per-round history replaces the stored mu
# entry afterwards.
eval_hist_fedprox = []
_eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedprox)

sname = f"fedprox_mlp_mu{RETRY_MU}"
strategy = CheckpointFedProx(sname, MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    proximal_mu=RETRY_MU, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp())))
strategy.model_dir = Path(MODEL_DIR) / sname

print(f"\n=== FedProx MLP (mu={RETRY_MU}) ===")
fl.simulation.start_simulation(
    client_fn=mlp_client_fn, num_clients=4, config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedprox_histories[RETRY_MU] = eval_hist_fedprox.copy()
print(f"FedProx mu={RETRY_MU} done. {len(eval_hist_fedprox)} rounds.")
last = eval_hist_fedprox[-1] if eval_hist_fedprox else {}
print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")
print(f"  Best All_BalAcc:  {max((h.get('All_BalAcc',0) for h in eval_hist_fedprox[1:]), default=0):.4f}")

In [ ]:
# Assembles the updated comparison table: baselines and other
# strategies are reused from the reloaded histories, and the
# retried mu row takes its peak pooled-BalAcc round (round 1
# excluded), then prints it.
def best_metrics(h):
    """Pick the peak pooled-BalAcc round from a per-round metric history.

    Parameters
    ----------
    h : list of dict
        Per-round records with ``round``, ``{site}_BalAcc``/``_AUC`` and
        ``All_BalAcc``/``All_AUC`` keys.

    Returns
    -------
    tuple
        ``(record, round_number)`` of the best round after round 1, or
        ``({}, 0)`` when the history is empty. Round 1 is skipped because it
        reflects the zero-initialised global model.
    """
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    """Append one method's per-site and pooled metrics to ``rows``.

    Parameters
    ----------
    method : str
        Row label of the result table.
    cs_source : dict, optional
        Non-federated baseline metrics (centralized / cross-site) to copy.
    fed_hist : list of dict, optional
        Federated per-round history; its peak round supplies the values.
    """
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan); r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan); r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round; rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
if fedavg_hist: add_row("FL FedAvg (MLP)", fed_hist=fedavg_hist)
if RETRY_MU in fedprox_histories:
    add_row(f"FL FedProx mu={RETRY_MU} (MLP)", fed_hist=fedprox_histories[RETRY_MU])
if fedlr_hist: add_row("FL FedAvg (LR)", fed_hist=fedlr_hist)
if fedrf_hist: add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
if cross_site_results:
    r = {"Method":"Cross-Site MLP","Label":"Cross-Site MLP","Peak_Round":0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_results.get(f"{site}_BalAcc",np.nan)
        r[f"{site}_AUC"] = cross_site_results.get(f"{site}_AUC",np.nan)
    r["All_BalAcc"] = cross_site_results.get("All_BalAcc",np.nan); r["All_AUC"] = cross_site_results.get("All_AUC",np.nan)
    rows.append(r)
if cross_site_rf_results:
    r = {"Method":"Cross-Site RF","Label":"Cross-Site RF","Peak_Round":0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_rf_results.get(f"{site}_BalAcc",np.nan)
        r[f"{site}_AUC"] = cross_site_rf_results.get(f"{site}_AUC",np.nan)
    r["All_BalAcc"] = cross_site_rf_results.get("All_BalAcc",np.nan); r["All_AUC"] = cross_site_rf_results.get("All_AUC",np.nan)
    rows.append(r)

df_results = pd.DataFrame(rows)
cols = ["Method","Label","Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# Two-panel convergence figure: FedAvg MLP per-site curves on the
# left, pooled BalAcc of every strategy (including the retried mu
# with its peak-round annotation) against the centralized
# baselines on the right; saved as results/convergence.pdf.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
if fedavg_hist:
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
        ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots_data = []
if fedavg_hist: plots_data.append(("FedAvg MLP", fedavg_hist, "#ff7f0e", "-"))
if RETRY_MU in fedprox_histories:
    plots_data.append((f"FedProx mu={RETRY_MU}", fedprox_histories[RETRY_MU], "#d62728", "--"))
if fedlr_hist: plots_data.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
if fedrf_hist: plots_data.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots_data:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
    skip0 = hist[1:]
    if skip0:
        best = max(skip0, key=lambda h: h.get("All_BalAcc", 0))
        pr = int(best.get("round", 0))
        ax.axvline(pr, color=c, ls=':', alpha=0.4, lw=1)
        ax.annotate(f"r{pr}", (pr, best.get("All_BalAcc", 0)), textcoords="offset points", xytext=(3,5), fontsize=7, color=c)
if centralized_mlp_results.get("All_BalAcc"):
    ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP')
if centralized_rf_results.get("All_BalAcc"):
    ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2, label='Centralized RF')
ax.set_title(f"All Methods (mu={RETRY_MU})"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
fig.suptitle(f"{DRUG_NAME} x {SPECIES} — FL Convergence", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Per-site FedProx mu curve for the retried value, with the
# centralized MLP pooled BalAcc as a reference line; saved as
# results/convergence_fedprox.pdf.
if RETRY_MU in fedprox_histories:
    fig, ax = plt.subplots(figsize=(8, 5))
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedprox_histories[RETRY_MU]]
        ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedprox_histories[RETRY_MU]]
    ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
    if centralized_mlp_results.get("All_BalAcc"):
        ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=1.5)
    ax.set_title(f"{DRUG_NAME} — FedProx mu={RETRY_MU} Per-Site"); ax.set_xlabel("Round")
    ax.set_ylabel("BalAcc"); ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)
    plt.tight_layout(); plt.savefig(OUT_DIR / "convergence_fedprox.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Per-method x per-site Balanced Accuracy heatmap of the updated
# table (All = pooled four-site value); saved as
# results/heatmap_balacc.pdf.
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)): ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_ba_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — BalAcc (mu={RETRY_MU})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Same heatmap layout for threshold-free AUC-ROC; saved as
# results/heatmap_auc.pdf.
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)): auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_auc_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — AUC (mu={RETRY_MU})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Writes the updated final_results.csv and the retried mu per-round
# CSV, and archives a copy of this notebook next to the results
# for provenance.
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
if RETRY_MU in fedprox_histories:
    pd.DataFrame(fedprox_histories[RETRY_MU]).to_csv(
        OUT_DIR / f"fedprox_mu{RETRY_MU}_per_round.csv", index=False)
    print(f"Overwrote fedprox_mu{RETRY_MU}_per_round.csv")

import shutil
src_nb = Path("retry_fedprox.ipynb")
if not src_nb.exists(): src_nb = Path.cwd() / "retry_fedprox.ipynb"
if not src_nb.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/retry_fedprox.ipynb", recursive=True))
    if candidates: src_nb = Path(candidates[0])
if src_nb.exists(): shutil.copy(str(src_nb), str(OUT_DIR / "retry_notebook.ipynb"))

print(f"\n{'='*60}")
print(f"  Done. Results updated in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

---
**Done.** FedProx retry complete.